In [1]:
from src.data_utils.corpus import Corpus
import numpy as np

In [2]:
fiqa = Corpus(dataset_name= 'fiqa', base_dir= 'data')

In [25]:
def bm25(dataset, query, k1=1, b=0.75): # Computes BM25 score for a given query in a set of documents

    bm25_score = {}

    import re
    query = re.sub(r"[^a-z ']+", " ", query.lower()) # clean up query. Makes everything lower case and removes anything that is not a lowercase letter, a space, or an apostrophe with a space.
    query = set(query.split()) # Split into unique words

    # Calculate term frequence (TF) matrix for all documents. Each document is a row with the frequencies of each term from the query in said document.
    TF_matrix = np.zeros((len(dataset), len(query)))
    for term in query:
        for i in range(len(dataset)):
            text = (dataset[i]['text'][0]) # load text
            text = re.sub(r"[^a-z ']+", " ", text.lower()) # clean up text, like with the query
            TF_matrix[i][list(query).index(term)] = text.count(term) # Tells us how many times each word appears in each document.
    
    # Calculate document frequency (DF) matrix. Use the fact that True=1, False=0 and sum over the rows.
    DF_matrix = (TF_matrix != 0).sum(axis=0) # tells us how many documents each word appears in.

    # Calculate average document length:
    doc_length = 0
    for k in range(len(dataset)):
        text = (dataset[k]['text'][0]) # load text
        text = re.sub(r"[^a-z ']+", " ", text.lower()) # clean up text, like with the query
        doc_length += len(text.split())
    avg_doc_length = doc_length/len(dataset)

    for j in range(len(dataset)):
        text = (dataset[j]['text'][0]) # load text
        text = re.sub(r"[^a-z ']+", " ", text.lower()) # clean up text, like with the query
        t = len(text.split())/avg_doc_length # t is used in the updated TF equation

        bm25 = 0
        for term in query:
            # Calculate f and DF for a given term.
            f = TF_matrix[j:j+1, list(query).index(term)][0]
            DF = DF_matrix[list(query).index(term)]

            # Do the inverse (I) calculation on the DF matrix.
            if DF == 0:
                IDF = 0 # to avoid division by 0
            else:
                IDF = np.log((len(dataset)+1) / (DF+0.5)) # updated IDF calculation
                
            bm25 += IDF * (f*(k1+1)) / (f+k1*(1-b+b*t)) # updated TF calculation

        bm25_score[f'Document {j+1}'] = float(bm25)
    
    return sorted(bm25_score.items(), key=lambda item: item[1], reverse=True)

bm25(fiqa.val, 'Obama used to be president.')


[('Document 192', 3.6379373356150677),
 ('Document 318', 3.5594914507917452),
 ('Document 305', 3.5209802276667106),
 ('Document 177', 3.2050427027188966),
 ('Document 307', 3.146874758664503),
 ('Document 486', 3.135366243310619),
 ('Document 340', 3.1092446875601634),
 ('Document 293', 3.087747633802171),
 ('Document 392', 3.082121196018551),
 ('Document 240', 3.060102751225981),
 ('Document 13', 2.9560181104223364),
 ('Document 65', 2.9482918141773147),
 ('Document 369', 2.9368367508185775),
 ('Document 335', 2.898888045741159),
 ('Document 497', 2.8799490039865865),
 ('Document 27', 2.8125526062815274),
 ('Document 165', 2.7653723992693227),
 ('Document 266', 2.735913457059172),
 ('Document 444', 2.6251209106126256),
 ('Document 330', 2.5557008235011733),
 ('Document 67', 2.5528129012337373),
 ('Document 302', 2.5364098453205974),
 ('Document 247', 2.52968901920113),
 ('Document 477', 2.5285277166873534),
 ('Document 341', 2.5223763683889233),
 ('Document 277', 2.5021265794713217),